In [ ]:
# ============================================================
# GOOGLE COLAB SETUP — run this cell first when using Colab
# ============================================================
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_PATH = '/content/drive/MyDrive/Factor-Research'

    if not os.path.exists(REPO_PATH):
        print('Cloning repository to Google Drive...')
        os.system(f'git clone https://github.com/mbrennan5/Factor-Research.git {REPO_PATH}')
    else:
        print(f'Repository found at {REPO_PATH}')

    print('Installing packages...')
    os.system('pip install -q lightgbm xgboost optuna plotly tqdm yfinance alpaca-py pyarrow')

    NOTEBOOKS_DIR = os.path.join(REPO_PATH, 'notebooks')
    os.chdir(NOTEBOOKS_DIR)
    print(f'Working directory set to: {os.getcwd()}')
else:
    print('Running locally — no Colab setup needed.')


# Complete Alpha Factor Selection Pipeline

## Pipeline Steps:

1. **IC Screening**: Filter factors with |IC| > 0.02
2. **Spearman Correlation Filtering**: Remove highly correlated factors (< 0.7)
3. **Lasso/Ridge Regression**: Further feature selection using regularization
4. **LGBM Feature Selection**: Tree-based feature importance selection
5. **Final Integration**: Combine results and save to `selected_factors/`

This comprehensive pipeline ensures high-quality, diversified factors for quantitative trading.


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso
from tqdm import tqdm
import glob
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

print("Libraries imported successfully")


In [ ]:
# === 1. Setup and Load Candidate Factors ===

# Import additional libraries for complete pipeline
from scipy.stats import spearmanr
import lightgbm as lgb
from sklearn.linear_model import Lasso, Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

# Path configuration
candidate_path = "../data/factors/obtained_features"  # Input: all candidate factors
selected_path = "../data/factors/selected_factors"    # Output: final selected factors

# Create output directory
os.makedirs(selected_path, exist_ok=True)

# Load all candidate alpha factor files
all_files = sorted(glob.glob(candidate_path + "/*.csv"))
print(f"Found {len(all_files)} candidate alpha factors")

# Load the return data (vwap1pct) as the prediction target
ret_path = '../data/processed/wide_data_preparation/vwap1pct_daily_data.csv'   
ret_wide = pd.read_csv(ret_path, index_col=0).fillna(0)
print(f"Return data shape: {ret_wide.shape}")

# Create date mapping for later use
date_map = {}
for i in range(len(ret_wide.index)):
    date_map[i] = ret_wide.index[i]
    
print(f"Date range: {ret_wide.index[0]} to {ret_wide.index[-1]}")
print(f"\n🚀 Starting Complete Alpha Factor Selection Pipeline...")


In [ ]:
# === 2. Step 1: IC-based Factor Screening ===

print("🔍 Step 1: IC Screening (|IC| > 0.02)")

# Initialize storage
ic_selected_factors = {}
ic_selected_names = []
factor_ic_stats = {}

ic_threshold = 0.02
max_ic = 0

for file in tqdm(all_files, desc="IC screening"):
    factor_name = file.split('/')[-1].replace('.csv', '')
    
    try:
        df = pd.read_csv(file, index_col=0)
        
        # Clean data
        if len(df.columns) > 0 and df.columns[0] == 'datetime_x':
            df.drop(df.columns[0], axis=1, inplace=True)
        
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.fillna(0, inplace=True)
        
        # Cross-sectional standardization
        stds = df.std(axis=1)
        df = df[stds != 0]
        
        if df.empty:
            continue
            
        df = df.sub(df.mean(axis=1), axis=0).div(df.std(axis=1), axis=0).fillna(0)
        
        # Calculate IC series
        ic_series = []
        valid_indices = df.index[df.index < len(ret_wide) - 1]
        
        for idx in valid_indices:
            factor_row = df.loc[idx]
            try:
                next_date = ret_wide.index[idx + 1]
                future_ret = ret_wide.loc[next_date]
                
                merged = pd.DataFrame({
                    'factor': factor_row,
                    'ret': future_ret
                }).dropna()
                
                if len(merged) < 100:
                    continue
                
                ic = merged['factor'].corr(merged['ret'])
                if pd.notnull(ic):
                    ic_series.append(ic)
            except Exception:
                continue
        
        if len(ic_series) == 0:
            continue
        
        # Calculate IC statistics
        mean_ic = np.mean(ic_series)
        mean_abs_ic = np.abs(mean_ic)
        ic_std = np.std(ic_series)
        ic_ir = mean_ic / (ic_std + 1e-8)
        
        if mean_abs_ic > max_ic:
            max_ic = mean_abs_ic
        
        # IC selection criteria
        if mean_abs_ic >= ic_threshold and df.index[0] <= 90:
            ic_selected_factors[factor_name] = df
            ic_selected_names.append(factor_name)
            factor_ic_stats[factor_name] = {
                'mean_ic': mean_ic,
                'mean_abs_ic': mean_abs_ic,
                'ic_std': ic_std,
                'ic_ir': ic_ir,
                'ic_count': len(ic_series)
            }
    
    except Exception as e:
        continue

print(f"\n📊 IC Screening Results:")
print(f"   - Candidate factors: {len(all_files)}")
print(f"   - Passed IC screening: {len(ic_selected_names)}")
print(f"   - Selection rate: {len(ic_selected_names)/len(all_files)*100:.1f}%")
print(f"   - Max |IC| observed: {max_ic:.4f}")


In [ ]:
# === 3. Step 2: Spearman Correlation Filtering ===

print("🔗 Step 2: Spearman Correlation Filtering (< 0.7)")

# Align factors to common dates and stocks
common_dates = set(range(len(ret_wide.index)))
for df in ic_selected_factors.values():
    common_dates &= set(df.index)
common_dates = sorted(list(common_dates))

common_stocks = set(ret_wide.columns)
for df in ic_selected_factors.values():
    common_stocks &= set(df.columns)
common_stocks = sorted(list(common_stocks))

print(f"Common dates: {len(common_dates)}, Common stocks: {len(common_stocks)}")

# Align factors
aligned_factors = {}
for name, df in ic_selected_factors.items():
    aligned_factors[name] = df.loc[common_dates, common_stocks]

# Calculate Spearman correlation matrix
factor_names = list(aligned_factors.keys())
n_factors = len(factor_names)
n_dates = len(common_dates)
n_stocks = len(common_stocks)

print(f"Computing Spearman correlations for {n_factors} factors...")

# Build data cube (dates, factors, stocks)
data_cube = np.array([aligned_factors[name].values for name in factor_names])
data_cube = np.transpose(data_cube, (1, 0, 2))

# Calculate correlation matrix
spearman_corr_matrix = np.zeros((n_factors, n_factors))

for i in tqdm(range(n_factors), desc="Computing correlations"):
    for j in range(i, n_factors):
        if i == j:
            spearman_corr_matrix[i, j] = 1.0
        else:
            daily_corrs = []
            for t in range(n_dates):
                x = data_cube[t, i]
                y = data_cube[t, j]
                
                mask = ~np.isnan(x) & ~np.isnan(y)
                if np.sum(mask) >= 20:
                    try:
                        corr, _ = spearmanr(x[mask], y[mask])
                        if not np.isnan(corr):
                            daily_corrs.append(corr)
                    except:
                        continue
            
            if len(daily_corrs) > 0:
                mean_corr = np.mean(daily_corrs)
                spearman_corr_matrix[i, j] = spearman_corr_matrix[j, i] = mean_corr

# Greedy selection with correlation constraint
correlation_threshold = 0.7
abs_corr_matrix = np.abs(spearman_corr_matrix)

# Sort factors by IC quality
factor_scores = []
for name in factor_names:
    ic_stat = factor_ic_stats[name]
    factor_scores.append({
        'name': name,
        'mean_abs_ic': ic_stat['mean_abs_ic'],
        'ic_ir': ic_stat['ic_ir'],
        'quality': ic_stat['mean_abs_ic'] * abs(ic_stat['ic_ir'])
    })

factor_scores = sorted(factor_scores, key=lambda x: x['quality'], reverse=True)

# Greedy selection
spearman_selected_factors = []
selected_indices = []

for factor_info in factor_scores:
    factor_name = factor_info['name']
    factor_idx = factor_names.index(factor_name)
    
    # Check correlation with already selected factors
    can_select = True
    for selected_idx in selected_indices:
        if abs_corr_matrix[factor_idx, selected_idx] >= correlation_threshold:
            can_select = False
            break
    
    if can_select:
        selected_indices.append(factor_idx)
        spearman_selected_factors.append(factor_name)

print(f"\n📊 Spearman Correlation Filtering Results:")
print(f"   - Input factors: {len(factor_names)}")
print(f"   - After correlation filtering: {len(spearman_selected_factors)}")
print(f"   - Removed factors: {len(factor_names) - len(spearman_selected_factors)}")
print(f"   - Correlation threshold: {correlation_threshold}")


In [ ]:
# === 4. Lasso/Ridge Regression Feature Selection ===

from sklearn.linear_model import Lasso, Ridge
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error

print("🔍 Step 1: Lasso/Ridge Regression Feature Selection")

# Test different alpha values for Lasso and Ridge
alphas = [0.0001, 0.001, 0.01, 0.1, 1.0]
lasso_scores = []
ridge_scores = []

print("\nTesting regularization strengths...")
for alpha in alphas:
    # Lasso
    lasso = Lasso(alpha=alpha, max_iter=1000)
    lasso_cv = cross_val_score(lasso, X_all, y_all, cv=5, scoring='neg_mean_squared_error')
    lasso_scores.append(-lasso_cv.mean())
    
    # Ridge
    ridge = Ridge(alpha=alpha)
    ridge_cv = cross_val_score(ridge, X_all, y_all, cv=5, scoring='neg_mean_squared_error')
    ridge_scores.append(-ridge_cv.mean())
    
    print(f"Alpha {alpha:6.4f}: Lasso MSE={-lasso_cv.mean():.6f}, Ridge MSE={-ridge_cv.mean():.6f}")

# Select best alpha for each method
best_lasso_alpha = alphas[np.argmin(lasso_scores)]
best_ridge_alpha = alphas[np.argmin(ridge_scores)]

print(f"\n🏆 Best Lasso alpha: {best_lasso_alpha}")
print(f"🏆 Best Ridge alpha: {best_ridge_alpha}")

# Train final models with best alphas
lasso_final = Lasso(alpha=best_lasso_alpha, max_iter=1000)
ridge_final = Ridge(alpha=best_ridge_alpha)

lasso_final.fit(X_all, y_all)
ridge_final.fit(X_all, y_all)

# Get feature importance/coefficients
lasso_coefs = pd.Series(lasso_final.coef_, index=selected_names)
ridge_coefs = pd.Series(ridge_final.coef_, index=selected_names)

# Select features with non-zero Lasso coefficients
lasso_selected = lasso_coefs[lasso_coefs != 0]
print(f"\n📊 Lasso Results:")
print(f"   - Input factors: {len(selected_names)}")
print(f"   - Non-zero coefficients: {len(lasso_selected)}")
print(f"   - Sparsity: {(len(selected_names) - len(lasso_selected))/len(selected_names)*100:.1f}%")

# Select top Ridge features by absolute coefficient
ridge_selected = ridge_coefs.abs().sort_values(ascending=False).head(len(lasso_selected))
print(f"\n📊 Ridge Results:")
print(f"   - Selected top {len(ridge_selected)} factors by |coefficient|")

print(f"\n🔍 Top 10 Lasso-selected factors:")
for factor, coef in lasso_selected.abs().sort_values(ascending=False).head(10).items():
    print(f"   {factor}: {coef:.6f}")

print(f"\n🔍 Top 10 Ridge-selected factors:")
for factor, coef in ridge_selected.head(10).items():
    print(f"   {factor}: {coef:.6f}")


In [ ]:
# === 5. LGBM Feature Selection ===

import lightgbm as lgb
from sklearn.model_selection import train_test_split

print("🔍 Step 2: LGBM Feature Selection")

# Split data for LGBM training
X_train, X_val, y_train, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

# Train LGBM model
lgb_params = {
    'objective': 'regression',
    'metric': 'mse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42
}

# Create datasets
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Train model
lgb_model = lgb.train(
    lgb_params,
    train_data,
    valid_sets=[val_data],
    num_boost_round=1000,
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
)

# Get feature importance
lgb_importance = pd.Series(lgb_model.feature_importance(), index=selected_names)
lgb_importance = lgb_importance.sort_values(ascending=False)

# Select top features by LGBM importance
n_top_lgbm = min(len(lasso_selected), 50)  # Select similar number as Lasso
lgbm_selected = lgb_importance.head(n_top_lgbm)

print(f"\n📊 LGBM Results:")
print(f"   - Input factors: {len(selected_names)}")
print(f"   - Selected top {len(lgbm_selected)} factors by importance")
print(f"   - Validation MSE: {lgb_model.best_score['valid_0']['mse']:.6f}")

print(f"\n🔍 Top 10 LGBM-selected factors:")
for factor, importance in lgbm_selected.head(10).items():
    print(f"   {factor}: {importance:.0f}")


In [ ]:
# === 6. Combine and Finalize Feature Selection ===

print("🔄 Step 3: Combining Lasso/Ridge + LGBM Results")

# Get union of selected features from all methods
lasso_factors = set(lasso_selected.index)
ridge_factors = set(ridge_selected.index)
lgbm_factors = set(lgbm_selected.index)

# Different combination strategies
union_all = lasso_factors | ridge_factors | lgbm_factors
intersection_all = lasso_factors & ridge_factors & lgbm_factors
lasso_lgbm_union = lasso_factors | lgbm_factors
lasso_lgbm_intersection = lasso_factors & lgbm_factors

print(f"\n📊 Feature Selection Combination Results:")
print(f"   - Lasso selected: {len(lasso_factors)}")
print(f"   - Ridge selected: {len(ridge_factors)}")
print(f"   - LGBM selected: {len(lgbm_factors)}")
print(f"   - Union of all: {len(union_all)}")
print(f"   - Intersection of all: {len(intersection_all)}")
print(f"   - Lasso ∪ LGBM: {len(lasso_lgbm_union)}")
print(f"   - Lasso ∩ LGBM: {len(lasso_lgbm_intersection)}")

# Use Lasso ∪ LGBM as final selection (most comprehensive)
final_selected_factors = lasso_lgbm_union
final_factor_list = list(final_selected_factors)

print(f"\n🎯 Final Selection Strategy: Lasso ∪ LGBM")
print(f"   - Final factor count: {len(final_selected_factors)}")

# Verify Spearman correlation constraint is still satisfied
print(f"\n🔍 Verifying Spearman correlation constraint...")

# Load correlation matrix if available
corr_matrix_path = "../data/factors/final_correlation_matrix.csv"
if os.path.exists(corr_matrix_path):
    corr_matrix = pd.read_csv(corr_matrix_path, index_col=0)
    
    # Check correlations among final selected factors
    final_corr_matrix = corr_matrix.loc[final_factor_list, final_factor_list]
    
    # Get upper triangle correlations (excluding diagonal)
    upper_triangle = np.triu(np.abs(final_corr_matrix.values), k=1)
    max_corr = np.max(upper_triangle[upper_triangle > 0])
    high_corr_count = np.sum(upper_triangle > 0.7)
    
    print(f"   - Max correlation among final factors: {max_corr:.3f}")
    print(f"   - Correlations > 0.7: {high_corr_count}")
    
    if max_corr < 0.7:
        print(f"   ✅ All correlations < 0.7 constraint satisfied")
    else:
        print(f"   ⚠️  Some correlations > 0.7 detected")
else:
    print(f"   ⚠️  Correlation matrix not found, skipping verification")

print(f"\n🏆 FINAL SELECTED FACTORS ({len(final_selected_factors)}):")
for i, factor in enumerate(sorted(final_factor_list), 1):
    print(f"   {i:2d}. {factor}")

# Save final factor list
final_factors_df = pd.DataFrame({
    'factor_name': final_factor_list,
    'selection_method': ['Lasso+LGBM'] * len(final_factor_list)
})
final_factors_df.to_csv('../data/factors/final_selected_factors_list.csv', index=False)
print(f"\n💾 Final factor list saved to: ../data/factors/final_selected_factors_list.csv")


In [ ]:
# === 7. Final Integration and Save to selected_factors ===

print("💾 Step 5: Final Integration and Save to selected_factors/")

# Use Lasso ∪ LGBM as final selection
final_selected_factors = lasso_factors | lgbm_factors
final_factor_list = list(final_selected_factors)

print(f"\n🎯 Final Selection Results:")
print(f"   - IC screening: {len(ic_selected_names)} factors")
print(f"   - Spearman filtering: {len(spearman_selected_factors)} factors")
print(f"   - Lasso selection: {len(lasso_factors)} factors")
print(f"   - LGBM selection: {len(lgbm_factors)} factors")
print(f"   - Final selection (Lasso ∪ LGBM): {len(final_selected_factors)} factors")

# Verify correlation constraint
print(f"\n🔍 Verifying final factors meet Spearman < 0.7 constraint...")
final_indices = [factor_names.index(name) for name in final_factor_list if name in factor_names]
if len(final_indices) > 1:
    final_corr_matrix = spearman_corr_matrix[np.ix_(final_indices, final_indices)]
    upper_triangle = np.triu(np.abs(final_corr_matrix), k=1)
    max_corr = np.max(upper_triangle[upper_triangle > 0]) if len(upper_triangle[upper_triangle > 0]) > 0 else 0
    high_corr_count = np.sum(upper_triangle > 0.7)
    
    print(f"   - Max correlation among final factors: {max_corr:.3f}")
    print(f"   - Correlations > 0.7: {high_corr_count}")
    
    if max_corr < 0.7:
        print(f"   ✅ All correlations < 0.7 constraint satisfied")
    else:
        print(f"   ⚠️  Some high correlations detected")

# Save final selected factors to selected_factors directory
print(f"\n💾 Saving {len(final_selected_factors)} factors to {selected_path}/...")

saved_count = 0
for factor_name in final_factor_list:
    if factor_name in aligned_factors:
        factor_data = aligned_factors[factor_name]
        
        # Convert index back to date strings for consistency
        factor_data.index = [ret_wide.index[i] for i in factor_data.index]
        
        # Save to selected_factors directory
        factor_data.to_csv(f"{selected_path}/{factor_name}.csv")
        saved_count += 1

print(f"✅ Successfully saved {saved_count} factors to {selected_path}/")

# Create final summary
final_summary = []
for name in final_factor_list:
    if name in factor_ic_stats:
        ic_stat = factor_ic_stats[name]
        method = []
        if name in lasso_factors:
            method.append('Lasso')
        if name in lgbm_factors:
            method.append('LGBM')
        
        final_summary.append({
            'factor_name': name,
            'mean_ic': ic_stat['mean_ic'],
            'mean_abs_ic': ic_stat['mean_abs_ic'],
            'ic_ir': ic_stat['ic_ir'],
            'selection_method': '+'.join(method)
        })

final_summary_df = pd.DataFrame(final_summary)
final_summary_df = final_summary_df.sort_values('mean_abs_ic', ascending=False)
final_summary_df.to_csv(f'{selected_path}/factor_summary.csv', index=False)

print(f"✅ Factor summary saved to {selected_path}/factor_summary.csv")

print(f"\n🏆 FINAL SELECTED FACTORS ({len(final_selected_factors)}):")
for i, factor in enumerate(sorted(final_factor_list)[:20], 1):  # Show top 20
    print(f"   {i:2d}. {factor}")
if len(final_factor_list) > 20:
    print(f"   ... and {len(final_factor_list)-20} more factors")

print(f"\n🎉 Complete Alpha Factor Selection Pipeline Finished!")
print(f"📁 Results saved in: {selected_path}/")
print(f"📊 Total selected factors: {len(final_selected_factors)}")


In [ ]:
# === 6. Evaluate Alpha Factor Model Performance ===

# Select a stock for detailed analysis (11th stock in the dataset)
stock_name = pred_df.columns[10] 
print(f"📊 Performance Analysis for Stock: {stock_name}")

# Get predicted returns for selected stock
pred_series = pred_df[stock_name]

# Get actual returns for selected stock
true_series = pd.Series(
    [ret_wide.loc[date_map[dt], stock_name] for dt in pred_df.index],
    index=pred_df.index
)

# Align series indices
pred_series = pred_series.loc[true_series.index]

# Calculate key performance metrics
correlation = pred_series.corr(true_series)
pred_mean, actual_mean = pred_series.mean(), true_series.mean()
pred_std, actual_std = pred_series.std(), true_series.std()

print(f"\\n🎯 Alpha Factor Model Performance Metrics:")
print(f"   - Correlation (Pred vs Actual): {correlation:.4f}")
print(f"   - Predicted return mean: {pred_mean:.6f}")
print(f"   - Actual return mean: {actual_mean:.6f}")
print(f"   - Predicted return std: {pred_std:.6f}")
print(f"   - Actual return std: {actual_std:.6f}")

# Create comprehensive visualization
plt.figure(figsize=(15, 10))

# Main time series plot
plt.subplot(2, 2, (1, 2))
plt.plot(pred_series.index, pred_series, label="Alpha Model Prediction", 
         marker='o', linestyle='--', alpha=0.7, markersize=2, color='red')
plt.plot(true_series.index, true_series, label="Actual Return", 
         marker='x', linestyle='-', alpha=0.8, markersize=3, color='blue')

plt.xlabel("Date")
plt.ylabel("Return Rate")
plt.title(f"Alpha Factor Model: Predicted vs Actual Returns\\nStock: {stock_name} | Correlation: {correlation:.4f}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

# Scatter plot
plt.subplot(2, 2, 3)
plt.scatter(pred_series, true_series, alpha=0.6, s=20)
plt.xlabel("Predicted Returns")
plt.ylabel("Actual Returns")
plt.title("Prediction Scatter Plot")
plt.grid(True, alpha=0.3)

# Add diagonal line for perfect prediction
min_val = min(pred_series.min(), true_series.min())
max_val = max(pred_series.max(), true_series.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8, label='Perfect Prediction')
plt.legend()

# Residuals plot
plt.subplot(2, 2, 4)
residuals = true_series - pred_series
plt.plot(residuals.index, residuals, marker='o', linestyle='-', alpha=0.7, markersize=2)
plt.xlabel("Date")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title(f"Prediction Residuals\\nMean: {residuals.mean():.6f}, Std: {residuals.std():.6f}")
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print(f"\\n🎉 Alpha Factor Selection and Model Training Completed Successfully!")
print(f"\\n📋 Final Summary:")
print(f"   - Selected {len(selected_names)} high-quality alpha factors from {len(all_files)-1} candidates")
print(f"   - Trained Lasso model with {len(non_zero_coefs)} active coefficients")
print(f"   - Achieved {correlation:.4f} correlation on sample stock {stock_name}")
print(f"   - Model ready for quantitative trading strategy implementation")
